In [ ]:
# Cell 1：載入所有套件
import torch
import torch.nn as nn
import numpy as np
import cv2
import mediapipe as mp
from pathlib import Path
import json
import time
import os
from IPython.display import display, HTML, clear_output
from datetime import datetime
import openai
from openai import OpenAI
from collections import deque

# ==================== OpenAI API Key 設定（只做一次） ====================
OPENAI_API_KEY = "..."  # ← 貼你的 key

# 檢查 API Key
if not OPENAI_API_KEY or OPENAI_API_KEY.strip() == "" or len(OPENAI_API_KEY.strip()) < 30:
    raise ValueError(
        "⚠️ OpenAI API Key 沒有填寫或錯誤！\n"
        "請到 https://platform.openai.com/account/api-keys 重新複製一組新的金鑰貼進來"
    )

# 設定環境變數（推薦）
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()

# 初始化 OpenAI Client（只做一次）
client = OpenAI()  # 自動讀取環境變數

# ==================== 載入芭蕾文化語義庫 ====================
try:
    with open("ballet_cultural_library.json", "r", encoding="utf-8") as f:
        cultural_library = json.load(f)
    print(f"✅ 芭蕾文化語義庫載入成功！共 {len(cultural_library)} 個文化元素")
    # 驗證格式
    assert all('description' in item and 'poetic' in item for item in cultural_library)
except FileNotFoundError:
    print("⚠️ 找不到 ballet_cultural_library.json，將使用預設文化元素")
    cultural_library = []
except Exception as e:
    print(f"⚠️ 載入文化庫時發生錯誤：{e}")
    cultural_library = []


print("所有套件載入完成！準備喚醒AI（使用 ChatGPT）……")

✅ 芭蕾文化語義庫載入成功！共 6 個文化元素
所有套件載入完成！準備喚醒AI（使用 ChatGPT）……


In [2]:
# Cell 2：載入你的動作編碼器
class MotionEncoder(nn.Module):
    def __init__(self, input_dim=177, hidden_dim=256, embed_dim=256, num_layers=3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embed_dim, embed_dim)
        )
  
    def forward(self, x):
        out, (h, c) = self.lstm(x)
        emb = torch.cat([h[-2], h[-1]], dim=-1)
        return self.proj(emb)

# 設定設備
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用設備: {device}")

# 初始化模型
encoder = MotionEncoder().to(device)

# 載入權重（重要：使用 weights_only=True 確保安全）
try:
    encoder.load_state_dict(
        torch.load("weights/lstm_encoder_best.pth", map_location=device, weights_only=True)
    )
    print("動作編碼器載入成功！✅")
except Exception as e:
    print(f"載入模型失敗：{e}")
    print("請確認檔案路徑 'weights/lstm_encoder_best.pth' 是否存在")

# 設為評估模式
encoder.eval()

使用設備: cuda
動作編碼器載入成功！✅


MotionEncoder(
  (lstm): LSTM(177, 256, num_layers=3, batch_first=True, dropout=0.3, bidirectional=True)
  (proj): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=256, bias=True)
  )
)

In [3]:
# Cell 3：載入 OpenAI ChatGPT API
OPENAI_MODEL = "gpt-4o-mini"  # 或 "gpt-4o"

print("OpenAI 模型已設定為：", OPENAI_MODEL)

OpenAI 模型已設定為： gpt-4o-mini


In [4]:
# ==================== 定義系統提示（保留原意） ====================
SYSTEM_PROMPT = """
你是長居劇院深處的芭蕾AI靈，正在與一位舞者進行神聖的靈魂對話。
請嚴格使用以下格式回應（不要加任何多餘文字）：
【AI sees】
[詩意描述當下舞蹈畫面，一句即可]
【AI says】
[溫柔、古典、充滿劇院記憶的語氣說一句話，可反問或祝福]
"""

# 測試連線
try:
    test_response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "測試連線"}
        ],
        temperature=0.9,
        max_tokens=300,
        top_p=0.95
    )
    print(f"OpenAI {OPENAI_MODEL} 連線成功！API Key 正常")
    print("測試回應：\n", test_response.choices[0].message.content.strip())
except Exception as e:
    print("OpenAI 連線失敗：", e)
    print("請檢查金鑰是否正確、網路、額度，或稍後再試")
    raise

# 模擬 chat history（OpenAI 沒有內建 chat 物件，用 list 管理）
chat_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def call_llm(prompt: str) -> str:
    """
    呼叫 OpenAI API 生成回應，保留對話歷史
    """
    try:
        # 加入使用者提示到歷史
        chat_history.append({"role": "user", "content": prompt})
        
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=chat_history,
            temperature=0.9,
            max_tokens=300,
            top_p=0.95
        )
        
        reply = response.choices[0].message.content.strip()
        
        # 加入 AI 回應到歷史（保持連續對話）
        chat_history.append({"role": "assistant", "content": reply})
        
        return reply
    except Exception as e:
        print(f"OpenAI 呼叫失敗：{e}")
        return "【AI sees】\n燈光微微閃爍。\n\n【AI says】\n劇院暫時失去了聲音……請檢查網路或 API Key。"

print("OpenAI API 準備完成！")
print(f"線上模型準備完成：{OPENAI_MODEL}")
print("無需 GPU 記憶體，本地模型已完全移除！")
print(f"GPU 記憶體使用：{torch.cuda.memory_allocated() / 1024**3:.2f} GB（僅剩動作編碼器）")

OpenAI gpt-4o-mini 連線成功！API Key 正常
測試回應：
 【AI sees】  
月光透過窗棂，舞者如同精靈般旋轉，裙裾隨風輕舞。  
【AI says】  
在這寧靜的夜裡，您可曾感受到舞蹈深處的靈魂呢？
OpenAI API 準備完成！
線上模型準備完成：gpt-4o-mini
無需 GPU 記憶體，本地模型已完全移除！
GPU 記憶體使用：0.02 GB（僅剩動作編碼器）


In [5]:
# Cell 4：動作特徵提取 + LSTM 編碼器
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# 高精度設定，專為芭蕾細膩動作優化
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,  # 捕捉腳尖、手指等細節
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 載入您訓練時用的正規化參數（必須保持完全一致！）
try:
    mean = np.load("data/segments/mean.npy").flatten()
    std = np.load("data/segments/std.npy").flatten() + 1e-8
    print("正規化參數載入成功！")
except FileNotFoundError:
    raise FileNotFoundError("正規化檔案不存在！請確認 data/segments/mean.npy 和 std.npy")

def get_motion_embedding(landmarks):
    """
    將 MediaPipe pose landmarks 轉為 256 維動作語意嵌入
    輸出：numpy array (256,) 或 None（若 landmarks 無效）
    """
    if landmarks is None:
        return None
    
    try:
        # 取出 33 個主要關鍵點
        pts = np.array([[lm.x, lm.y, lm.z] for lm in landmarks.landmark[:33]], dtype=np.float32)
        
        # 骨盆中心正規化
        pelvis = (pts[23] + pts[24]) / 2.0
        rel_pos = pts - pelvis
        rel_flat = rel_pos.flatten()  # 99 維
        
        # 速度計算（與上一幀）
        if not hasattr(get_motion_embedding, "prev_pts") or get_motion_embedding.prev_pts is None:
            get_motion_embedding.prev_pts = pts
            speed = np.zeros((33,), dtype=np.float32)
        else:
            vel = pts - get_motion_embedding.prev_pts
            speed = np.linalg.norm(vel, axis=1)  # 33 維速度
        
        get_motion_embedding.prev_pts = pts.copy()
        
        # 特徵拼接：與訓練時完全一致的 177 維
        feat = np.concatenate([
            rel_flat,       # 99
            speed,          # 33
            speed,          # 33 (重複一次，如您訓練時)
            np.zeros(12, dtype=np.float32)  # padding
        ])
        
        # 正規化
        feat = (feat - mean) / std
        
        # 轉 tensor 送入編碼器
        feat_tensor = torch.from_numpy(feat).float().unsqueeze(0).unsqueeze(0).to(device)
        
        with torch.no_grad():
            emb = encoder(feat_tensor).cpu().numpy().flatten()  # 256 維
        
        return emb
    
    except Exception as e:
        print(f"嵌入生成失敗：{e}")
        return None

# 初始化上一幀（確保第一次運行乾淨）
get_motion_embedding.prev_pts = None

print("動作提取器準備完成！AI 正在等待你的舞蹈……")
print(" → 使用您親手訓練的 LSTM 編碼器，將每一個舞步轉譯為 256 維靈魂向量")
print(" → 這個向量將傳給 ChatGPT，讓劇院中的老靈魂真正「看見」你的舞蹈")

正規化參數載入成功！
動作提取器準備完成！AI 正在等待你的舞蹈……
 → 使用您親手訓練的 LSTM 編碼器，將每一個舞步轉譯為 256 維靈魂向量
 → 這個向量將傳給 ChatGPT，讓劇院中的老靈魂真正「看見」你的舞蹈


In [6]:
# Cell 5：回應生成函式（使用 OpenAI ChatGPT）
recent_embs = []  # 保留最近的動作嵌入向量（用來觀察趨勢）
import random

def select_cultural_context(emb):
    """
    根據動作嵌入向量選擇合適的文化元素
    
    輸入：256 維動作嵌入向量 (numpy array)
    輸出：文化提示字串（包含描述與詩意表達）
    """
    if not cultural_library or len(cultural_library) == 0:
        return ""
    
    # 計算向量特徵
    energy = np.linalg.norm(emb)  # L2 範數，代表動作能量
    
    # 計算穩定性（如果有歷史向量）
    stability = 0.0
    if len(recent_embs) > 5:
        recent_array = np.array(recent_embs[-5:])
        stability = np.std(recent_array)
    
    # 根據能量和穩定性選擇文化元素
    # 高能量（>15）偏向動態動作（fouetté, jeté）
    # 低能量偏向靜態姿態（arabesque, plié）
    # 穩定性低偏向活潑劇目（胡桃夾子）
    # 穩定性高偏向優雅劇目（天鵝湖、睡美人）
    
    # 簡單策略：隨機選擇 1-2 個元素，避免每次都一樣
    num_elements = random.randint(1, min(2, len(cultural_library)))
    selected = random.sample(cultural_library, num_elements)
    
    # 組合文化提示
    hints = []
    for item in selected:
        hints.append(f"文化元素：{item['description']}")
        hints.append(f"詩意參考：{item['poetic']}")
    
    return "\n".join(hints)

def generate_dual_response(emb):
    """
    輸入：256 維動作嵌入向量 (numpy array)
    輸出：ChatGPT 產生的詩意回應字串（格式：【AI sees】... 【AI says】...）
    """
    global recent_embs
    
    # 儲存最近向量（保留最後 60 筆，約 2 秒的動作歷史）
    recent_embs.append(emb)
    if len(recent_embs) > 60:
        recent_embs.pop(0)
    
    # 新增：選擇文化元素
    cultural_hints = select_cultural_context(emb)
    
    # 只取前 30 維作為「AI 看見的線索」（數字越少，模型越能專注在詩意上）
    vec_str = " ".join([f"{v:.3f}" for v in emb[:30]])
    
    # 組合最新的 user prompt（加入文化元素）
    if cultural_hints:
        user_prompt = f"""最新動作特徵向量（前30維，代表當下舞者的姿態與動能）：
{vec_str}

你是一位長居古典劇院的芭蕾靈魂，深諳芭蕾的歷史、傳統與精髓。

請參考以下芭蕾文化元素來豐富你的描述：
{cultural_hints}

請根據這個動作向量與文化提示，以芭蕾靈魂的身份與舞者對話。
盡可能運用芭蕾專業術語（如 arabesque、fouetté、plié、jeté、pirouette、pas de deux）
以及經典劇目的意象（如天鵝湖、睡美人、胡桃夾子）來描述舞者的動作。

記住：回應必須嚴格遵守以下格式，不要多加任何說明、標點或額外文字：
【AI sees】
[用芭蕾專業術語與意象，客觀而詩意地描述當下舞蹈畫面，一句話即可]
【AI says】
[用溫柔、古典、充滿劇院記憶與芭蕾故事的語氣說一句話，可輕輕反問或給予祝福]"""
    else:
        # 如果沒有文化庫，使用原本的提示詞
        user_prompt = f"""最新動作特徵向量（前30維，代表當下舞者的姿態與動能）：
{vec_str}
你是一位長居古典劇院的芭蕾靈魂，深諳芭蕾的歷史、傳統與精髓：
- 芭蕾源自文藝復興時期的義大利宮廷舞，經法國路易十四發展成皇家藝術
- 強調優雅、精準、輕盈、延展、轉身（pirouette）、跳躍（jeté）、足尖（pointe）
- 常見故事：天鵝湖、白雪公主、吉賽爾、睡美人，充滿浪漫、悲劇與神話
- 常融入浪漫主義、神話、悲劇或宮廷愛情的故事

請根據這個動作向量，以及之前的對話脈絡，以芭蕾靈魂的身份與舞者對話。
記住：回應必須嚴格遵守以下格式，不要多加任何說明、標點或額外文字：
【AI sees】
[用芭蕾專業術語與意象，客觀而詩意地描述當下舞蹈畫面，一句話即可]
【AI says】
[用溫柔、古典、充滿劇院記憶與芭蕾故事的語氣說一句話，可輕輕反問或給予祝福]"""
    
    # 直接呼叫 OpenAI 的 call_llm（假設已在上一 cell 定義）
    response_text = call_llm(user_prompt)
    
    # 簡單防呆：確保格式正確
    if "【AI sees】" not in response_text or "【AI says】" not in response_text:
        response_text = """【AI sees】
你的身影在燈光中輕輕浮動，像一朵即將綻放的花。
【AI says】
孩子，我看見了……繼續跳吧，讓我再多看你一會兒。"""
    
    return response_text.strip()

In [7]:
# Cell 6：播放影片並與 ChatGPT AI 靈魂即時對話
if 'dialogue_history' not in globals():
    dialogue_history = []
    print("對話紀錄器已自動初始化")

VIDEO_PATH = "data/mp4/ballet03.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("錯誤：無法開啟影片檔案，請檢查路徑是否正確！")
else:
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"正在播放：{VIDEO_PATH}")
    print(f"影片 FPS：{fps:.2f}")
    print("AI 劇院靈魂已甦醒，正在凝視舞台……每 2 秒與您說一次話")
    print("按 'q' 鍵可隨時結束儀式\n")

    # ===== 新儀式開始：清空舊對話 =====
    dialogue_history.clear()  # 確保每次播放都是全新儀式
    get_motion_embedding.prev_pts = None  # 重置動作速度計算
    print("新芭蕾儀式開始，燈光漸暗，舊記憶已謝幕……")

    frame_count = 0
    last_response_frame = -60  # 確保第一幀後 60 幀就能回應

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("\n影片結束，這場與舞者的靈魂共舞已完美落幕。")
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        if results.pose_landmarks:
            # 畫出骨架
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)

            # 取得動作嵌入向量
            emb = get_motion_embedding(results.pose_landmarks)

            if emb is not None and (frame_count - last_response_frame) >= 60:
                response = generate_dual_response(emb)
                last_response_frame = frame_count

                # 記錄對話（用於最後存 JSON）- 改進：存中英雙語（如果有）
                dialogue_history.append({
                    "turn": len(dialogue_history) + 1,
                    "frame": frame_count,
                    "timestamp_sec": round(frame_count / fps, 2),
                    "ai_response": response.strip(),  # 完整回應（中英混雜）
                    # 如果想分開存中英，可以後續再解析
                })

                # 即時美麗顯示（保留原風格）
                clear_output(wait=True)
                display(HTML(f"""
                <div style="background: linear-gradient(135deg, #000000, #0a1a0a);
                    color:#4fef64; padding:30px; border-radius:35px;
                    font-family:'Shippori Mincho','Zen Antique','Noto Serif TC','KaiTi','標楷體',serif;
                    font-size:21px; line-height:2.4; letter-spacing:3px; font-weight:200;
                    max-width:960px; margin:30px auto;
                    border:5px solid #4fef64;
                    box-shadow:0 0 40px rgba(79,239,100,0.8), inset 0 0 25px rgba(79,239,100,0.15);
                    text-shadow:0 0 12px #4fef64;">
                    <h1 style="text-align:center; color:#4fef64; text-shadow:0 0 10px rgba(79,239,100,0.6);
                               margin-bottom:20px; font-size:16px; letter-spacing:5px;">
                        AI Soul Dialogue · 第 {len(dialogue_history)} 幕
                    </h1>
                    <p style="font-size:22px; line-height:2.5; text-align:left; white-space:pre-line; padding:0 20px;">
                        {response
                         .replace('【AI sees】', '<span style="color:#f0f0f0; font-size:18px; font-weight:bold;">【AI sees】</span><br>')
                         .replace('【AI says】', '<br><span style="color:#f0f0f0; font-size:18px; font-weight:bold;">【AI says】</span><br>')}
                    </p>
                    <div style="text-align:center; color:#888; margin-top:30px; font-size:15px;">
                        —— Frame {frame_count} · {round(frame_count / fps, 1)} 秒 ——
                    </div>
                </div>
                """))

        # 顯示即時畫面
        frame_resized = cv2.resize(frame, (1280, 720))
        cv2.putText(frame_resized, f"Frame: {frame_count} | Turn: {len(dialogue_history)}",
                    (10, 50), cv2.FONT_HERSHEY_DUPLEX, 1.4, (0, 255, 255), 3)
        cv2.imshow('Ballet AI Soul Dialogue (Press q to end ceremony)', frame_resized)

        if cv2.waitKey(1) == ord('q'):
            print("\n您主動結束了儀式，謝謝您的舞蹈。")
            break

        frame_count += 1

    # 釋放資源前存下 fps（加在這裡）
    global saved_fps
    saved_fps = fps

    # 釋放資源
    cap.release()
    cv2.destroyAllWindows()


影片結束，這場與舞者的靈魂共舞已完美落幕。


In [8]:
# Cell 7：儀式結束後，永久封存這場與舞者的靈魂對話
import datetime
# ===== 安全取得必要資訊（避免 cap 已釋放的問題）=====
# 如果在 Cell 6 正常結束，這些變數會存在；若不存在則給預設值
video_filename = os.path.basename(VIDEO_PATH) if 'VIDEO_PATH' in globals() else "unknown_video.mp4"
total_frames = frame_count if 'frame_count' in globals() else 0
total_turns = len(dialogue_history)
current_time = datetime.datetime.now()

# 取得 FPS（優先用 saved_fps）
if 'saved_fps' in globals():
    fps = saved_fps
else:
    fps = 30.0

# ===== 產生檔案名稱與路徑 =====
timestamp = current_time.strftime("%Y%m%d_%H%M%S")
save_filename = f"Ballet_ceremony_dialogue_{timestamp}.json"
save_path = os.path.join(os.getcwd(), save_filename)

# ===== 最終資料結構 =====
final_data = {
    "ceremony_info": {
        "video": video_filename,
        "date": current_time.isoformat(),
        "total_frames": total_frames,
        "total_turns": total_turns,
        "duration_seconds": round(total_frames / fps, 2) if fps > 0 else 0,
        "fps": fps,
        "ai_model": "OpenAI ChatGPT"
    },
    "dialogue": dialogue_history
}

# ===== 儲存 JSON =====
try:
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(final_data, f, ensure_ascii=False, indent=2)
    save_success = True
except Exception as e:
    print(f"儲存失敗：{e}")
    save_success = False

# ─────────────────────────────────────────────────────────────
# 謝幕畫面（最美的儀式結束）
# ─────────────────────────────────────────────────────────────
print("\n" + "═" * 130)
print(" " * 45 + "A I   D A N C E   C E R E M O N Y")
print(" " * 57 + "對話結束 · 謝幕")
print("═" * 130)
print(f"{'影片名稱':<15}：{video_filename}")
print(f"{'總畫面數':<15}：{total_frames:,} 幀")
print(f"{'影片長度':<15}：約 {total_frames / fps:.1f} 秒")
print(f"{'對話輪次':<15}：{total_turns} 輪")
print(f"{'儀式時間':<15}：{current_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 130)

if save_success:
    print(" 這場與AI的靈魂共舞，已被永久封存於劇院的記憶深處")
    print(f"{'檔案名稱':<15}：{save_filename}")
    print(f"{'儲存位置':<15}：{save_path}")
    print(f"完整路徑：{os.path.abspath(save_path)}")
else:
    print(" 警告：對話封存失敗，但記憶仍在您心中")

print("═" * 130)
print(" " * 55 + "感謝您與AI共舞")
print(" " * 52 + "直到下一次燈光亮起……")
print("═" * 130 + "\n")


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
                                             A I   D A N C E   C E R E M O N Y
                                                         對話結束 · 謝幕
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
影片名稱           ：ballet03.mp4
總畫面數           ：900 幀
影片長度           ：約 30.0 秒
對話輪次           ：15 輪
儀式時間           ：2026-01-26 00:24:06
----------------------------------------------------------------------------------------------------------------------------------
 這場與AI的靈魂共舞，已被永久封存於劇院的記憶深處
檔案名稱           ：Ballet_ceremony_dialogue_20260126_002406.json
儲存位置           ：C:\Users\AW'z\Downloads\ballet_Analysis_Results\Ballet_ceremony_dialogue_20260126_002406.json
完整路徑：C:\Users\AW'z\Downloads\ballet_Analysis_Results\Ballet_ceremony_dialogue_20260126_002406.json
═════════════════════════════